In [30]:
import pyautogui as pyg
import time
import sys  # 【新增】用來立即強制結束程式

def main():
    print("【Chrome 小恐龍自動化腳本】啟動！")
    print("請在 5 秒內，將滑鼠游標移到「小恐龍嘴部的深色像素」上，不要移動...")
    time.sleep(5)
    
    dino_x, dino_y = pyg.position()
    print(f"✅ 基準點已鎖定: X={dino_x}, Y={dino_y}")
    print("遊戲即將開始！若要手動中斷，請將滑鼠狂掃至螢幕最「左上角」。")
    
    pyg.press('space')
    time.sleep(1)
    
    start_time = time.time()
    last_jump_time = 0  # 【新增】用來記錄最後一次跳躍的時間戳記
    last_obs_x = -1
    static_count = 0

    try:
        while True:
            # 手動檢查防呆機制：滑鼠移到左上角秒停
            mouse_x, mouse_y = pyg.position()
            if mouse_x <= 5 and mouse_y <= 5:
                print("\n🛑 已手動觸發左上角緊急停止，程式安全結束！")
                sys.exit(0) # 立即強制結束程式

            current_time = time.time()
            elapsed_time = current_time - start_time
            
            # 預判距離動態增長
            lookahead_dist = int(90 + (elapsed_time * 3.5))
            if lookahead_dist > 350: 
                lookahead_dist = 350
                
            # 偵測區域
            box_x = dino_x + 40
            box_y = dino_y - 5 
            box_w = lookahead_dist
            box_h = 30
            
            im = pyg.screenshot(region=(box_x, box_y, box_w, box_h))
            
            # 取第一點作為背景參考色
            bg_color = im.getpixel((0, 0))
            bg_r = bg_color[0]
            
            jump = False
            current_obs_x = -1
            
            # 步進掃描
            for x in range(0, box_w, 4):
                for y in range(0, box_h, 4):
                    r = im.getpixel((x, y))[0]
                    # 日夜色差門檻維持 40
                    if abs(r - bg_r) > 40:
                        jump = True
                        current_obs_x = x
                        break
                if jump:
                    break
            
            # 【優化】遊戲結束判定 (連續 5 次畫面凍結即判定撞擊)
            if current_obs_x != -1 and current_obs_x == last_obs_x:
                static_count += 1
            else:
                static_count = 0
                last_obs_x = current_obs_x
                
            if static_count >= 5:
                print(f"\n💀 偵測到撞擊！遊戲結束。共存活了 {elapsed_time:.1f} 秒。")
                sys.exit(0) # ⚡ 偵測到撞擊，不拖泥帶水，瞬間結束整個 Python 程式！
            
            # 執行跳躍
            if jump:
                cooldown = max(0.08, 0.40 - (elapsed_time * 0.0025))
                # 【關鍵修改】無阻塞冷卻 (Non-blocking)
                # 只有當「現在時間」減去「上次跳躍時間」大於冷卻時間時，才按下空白鍵
                # 這樣迴圈就不會被 time.sleep 卡住，截圖掃描的速度會達到極致！
                if current_time - last_jump_time > cooldown:
                    pyg.press('space')
                    last_jump_time = current_time

    except Exception as e:
        print(f"\n⚠️ 發生未預期的錯誤結束: {e}")
        sys.exit(1)

if __name__ == '__main__':
    main()

【Chrome 小恐龍自動化腳本】啟動！
請在 5 秒內，將滑鼠游標移到「小恐龍嘴部的深色像素」上，不要移動...
✅ 基準點已鎖定: X=1052, Y=392
遊戲即將開始！若要手動中斷，請將滑鼠狂掃至螢幕最「左上角」。

🛑 已手動觸發左上角緊急停止，程式安全結束！


SystemExit: 0